# Google Trends, HCUP, & U.S. Census Bureau Data Collection
**Author:** J. Casey Brookshier  
**Date:** July 15, 2026

## Objective

Collect, clean, and merge Google Trends, HCUP, and U.S. Census Bureau data to build a state-year dataset for predicting future inpatient psychiatric admissions.

### Data Sources
**Google Trends:** State-level suicide/crisis search interest (predictors)
  **HCUP:** Annual Mental Health/Substance Use inpatient admissions (outcome)
  **U.S. Census Bureau:** State population estimates (normalization)

### Workflow
**Collect → Clean → Merge → Model → Evaluate**



In [1]:
# imports & project paths

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().resolve().parents[1]

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Project root: /Users/caseybrookshier/Team-Rho-Capstone-Project
Raw data: /Users/caseybrookshier/Team-Rho-Capstone-Project/data/raw
Processed data: /Users/caseybrookshier/Team-Rho-Capstone-Project/data/processed


In [2]:
#define input output files

GT_RAW_FILE = RAW_DIR / "google_trends_raw_progress.csv"
HCUP_RAW_FILE = RAW_DIR / "DownloadTable_StatePayerIP_2026-03-20.xls"
CENSUS_2020_FILE = RAW_DIR / "nst-est2020.xlsx"
CENSUS_2025_FILE = RAW_DIR / "NST-EST2025-POP.xlsx"

OUTPUT_FILE = (
    PROCESSED_DIR
    / "team_rho_state_year_prediction_dataset.csv"
)

SEARCH_TERMS = [
    "suicidal thoughts",
    "suicide hotline",
    "self harm",
    "mental health crisis",
    "suicide prevention",
    "crisis hotline",
    "psychiatric hospital",
    "depression help",
]

YEARS = range(2013, 2024)

In [3]:
# verify repo structure

required_dirs = [
    RAW_DIR,
    PROCESSED_DIR,
]
# loop folders , raise error if missing
for directory in required_dirs:
    if not directory.exists():
        raise FileNotFoundError(f"Missing directory: {directory}")

print("Repository structure verified.")

Repository structure verified.


In [4]:
# load google trends data
# convert search values to numbers and averages by state-year

if not GT_RAW_FILE.exists():
    raise FileNotFoundError(
        f"Missing Google Trends file: {GT_RAW_FILE}"
    )

google = pd.read_csv(GT_RAW_FILE)

google["State"] = google["State"].astype(str).str.strip()
google["Year"] = pd.to_numeric(
    google["Year"],
    errors="coerce"
)

google = google[
    google["Year"].between(2013, 2023)
].copy()

google["Year"] = google["Year"].astype(int)

for term in SEARCH_TERMS:
    if term not in google.columns:
        google[term] = np.nan

    google[term] = pd.to_numeric(
        google[term],
        errors="coerce"
    )

google_year = (
    google
    .groupby(
        ["State", "Year"],
        as_index=False
    )[SEARCH_TERMS]
    .mean()
)

google_year = (
    google_year
    .sort_values(["State", "Year"])
    .reset_index(drop=True)
)

print("Google Trends shape:", google_year.shape)
print("States:", google_year["State"].nunique())
print("Years:", sorted(google_year["Year"].unique()))

Google Trends shape: (561, 10)
States: 51
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [5]:
#standarize google data
# clean fields , fill missing 
# search-term columns with NaN,remove duplicates.

google["State"] = google["State"].astype(str).str.strip()
google["Year"] = pd.to_numeric(google["Year"], errors="coerce")

google = google[
    google["Year"].between(2013, 2023)
].copy()

google["Year"] = google["Year"].astype(int)

for term in SEARCH_TERMS:
    if term not in google.columns:
        google[term] = np.nan

    google[term] = pd.to_numeric(
        google[term],
        errors="coerce"
    )

if "Search_Term" in google.columns:
    google_year = (
        google
        .groupby(["State", "Year"], as_index=False)[SEARCH_TERMS]
        .mean()
    )
else:
    google_year = google[
        ["State", "Year"] + SEARCH_TERMS
    ].copy()

google_year = (
    google_year
    .sort_values(["State", "Year"])
    .drop_duplicates(["State", "Year"])
    .reset_index(drop=True)
)

print("Google Trends shape:", google_year.shape)
print("States:", google_year["State"].nunique())
print("Years:", sorted(google_year["Year"].unique()))

Google Trends shape: (561, 10)
States: 51
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [6]:
# create google trend lag variables
# move state search value forward one year

for term in SEARCH_TERMS:
    google_year[f"{term}_lag1"] = (
        google_year
        .groupby("State")[term]
        .shift(1)
    )

google_year.head()

,State,Year,suicidal thoughts,suicide hotline,self harm,mental health crisis,suicide prevention,crisis hotline,psychiatric hospital,depression help,suicidal thoughts_lag1,suicide hotline_lag1,self harm_lag1,mental health crisis_lag1,suicide prevention_lag1,crisis hotline_lag1,psychiatric hospital_lag1,depression help_lag1
0,Alabama,2013,66.833333,11.166667,58.500000,0.000000,9.833333,0.0,30.166667,37.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alabama,2014,42.083333,14.500000,58.250000,0.000000,16.833333,0.0,32.166667,47.750000,66.833333,11.166667,58.500000,0.0,9.833333,0.0,30.166667,37.333333
2,Alabama,2015,47.250000,33.916667,49.916667,0.000000,24.333333,0.0,28.666667,43.083333,42.083333,14.500000,58.250000,0.0,16.833333,0.0,32.166667,47.750000
3,Alabama,2016,52.250000,38.583333,41.416667,0.000000,20.916667,0.0,45.583333,45.583333,47.250000,33.916667,49.916667,0.0,24.333333,0.0,28.666667,43.083333
4,Alabama,2017,50.833333,61.166667,33.666667,5.416667,29.166667,0.0,53.166667,52.333333,52.250000,38.583333,41.416667,0.0,20.916667,0.0,45.583333,45.583333


In [7]:
# load HCUP admission data, locate header row
# convert quarterly values to numbers
#select relevant years , group by state, sum quarters by year
# create  state year admission value


if not HCUP_RAW_FILE.exists():
    raise FileNotFoundError(
        f"Missing HCUP file: {HCUP_RAW_FILE}"
    )

raw_hcup = pd.read_excel(
    HCUP_RAW_FILE,
    sheet_name="Data",
    header=None
)

header_row = None

for i in range(len(raw_hcup)):
    row = raw_hcup.iloc[i].astype(str).tolist()

    if (
        "State" in row
        and "Hospitalization Type" in row
        and "Expected Payer" in row
    ):
        header_row = i
        break

if header_row is None:
    raise ValueError("HCUP header row not found.")

hcup = pd.read_excel(
    HCUP_RAW_FILE,
    sheet_name="Data",
    header=header_row
)

hcup.columns = (
    hcup.columns
    .astype(str)
    .str.strip()
)

quarter_cols = [
    c for c in hcup.columns
    if (
        isinstance(c, str)
        and len(c) >= 7
        and c[:4].isdigit()
        and "Q" in c
    )
]

for c in quarter_cols:
    hcup[c] = (
        hcup[c]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace(
            ["nan", "None", ""],
            np.nan
        )
    )

    hcup[c] = pd.to_numeric(
        hcup[c],
        errors="coerce"
    )

analysis_quarters = [
    c for c in quarter_cols
    if any(
        str(year) in c
        for year in YEARS
    )
]

hcup = hcup[
    [
        "State",
        "Hospitalization Type",
        "Expected Payer",
        *analysis_quarters
    ]
].copy()

mh = hcup[
    hcup["Hospitalization Type"]
    == "Mental Health/Substance Use"
].copy()

summary_patterns = (
    "Combined|"
    "Sum|"
    "All expected"
)

mh = mh[
    ~mh["Expected Payer"]
    .str.contains(
        summary_patterns,
        case=False,
        regex=True,
        na=False
    )
]

valid_payers = [
    "Medicare, age 65+",
    "Medicaid, age 19-64",
    "Private, age 19-64",
    "Self-Pay/No Charge, age 19-64"
]

mh = mh[
    mh["Expected Payer"].isin(valid_payers)
].copy()

mh_state_quarter = (
    mh
    .groupby("State")[analysis_quarters]
    .sum(min_count=1)
    .reset_index()
)

annual_records = []

for year in YEARS:

    year_cols = [
        c for c in analysis_quarters
        if str(year) in c
    ]

    temp = mh_state_quarter[
        ["State", *year_cols]
    ].copy()

    temp["Year"] = year

    temp["mental_health_admissions"] = (
        temp[year_cols]
        .sum(
            axis=1,
            min_count=1
        )
    )

    annual_records.append(
        temp[
            [
                "State",
                "Year",
                "mental_health_admissions"
            ]
        ]
    )

hcup = pd.concat(
    annual_records,
    ignore_index=True
)

hcup["State"] = (
    hcup["State"]
    .astype(str)
    .str.strip()
)

hcup["Year"] = hcup["Year"].astype(int)

hcup = hcup[
    hcup["mental_health_admissions"].notna()
    & (
        hcup["mental_health_admissions"] >= 0
    )
].copy()

hcup = (
    hcup
    .drop_duplicates(["State", "Year"])
    .sort_values(["State", "Year"])
    .reset_index(drop=True)
)

print("HCUP shape:", hcup.shape)
print("States:", hcup["State"].nunique())
print("Years:", sorted(hcup["Year"].unique()))

HCUP shape: (496, 3)
States: 46
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [8]:
# validate HCUP admissions data

if hcup.duplicated(["State", "Year"]).any():
    raise ValueError("HCUP contains duplicate State-Year rows.")

if hcup["mental_health_admissions"].isna().any():
    raise ValueError("HCUP contains missing admission values.")

if (hcup["mental_health_admissions"] < 0).any():
    raise ValueError("HCUP contains negative admission values.")

print("HCUP validation passed.")

HCUP validation passed.


In [9]:
# load census population data, clean state names,
# wide to long format, convert values to numbers

if not CENSUS_2020_FILE.exists():
    raise FileNotFoundError(
        f"Missing Census file: {CENSUS_2020_FILE}"
    )

if not CENSUS_2025_FILE.exists():
    raise FileNotFoundError(
        f"Missing Census file: {CENSUS_2025_FILE}"
    )

old_raw = pd.read_excel(
    CENSUS_2020_FILE,
    header=None
)

new_raw = pd.read_excel(
    CENSUS_2025_FILE,
    header=None
)

old_start = old_raw[
    old_raw.iloc[:, 0]
    .astype(str)
    .str.startswith(".Alabama")
].index[0]

new_start = new_raw[
    new_raw.iloc[:, 0]
    .astype(str)
    .str.startswith(".Alabama")
].index[0]

old_years = [
    "State",
    "2010",
    "Base",
    "2010",
    "2011",
    "2012",
    "2013",
    "2014",
    "2015",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020_Apr",
    "2020"
]

old = old_raw.iloc[old_start:].copy()
old.columns = old_years

old = old[
    [
        "State",
        "2013",
        "2014",
        "2015",
        "2016",
        "2017",
        "2018",
        "2019",
        "2020"
    ]
]

new_years = [
    "State",
    "Base",
    "2020",
    "2021",
    "2022",
    "2023",
    "2024",
    "2025"
]

new = new_raw.iloc[new_start:].copy()
new.columns = new_years

new = new[
    [
        "State",
        "2021",
        "2022",
        "2023"
    ]
]

wide = old.merge(
    new,
    on="State",
    how="left"
)

wide["State"] = (
    wide["State"]
    .astype(str)
    .str.replace(
        ".",
        "",
        regex=False
    )
    .str.strip()
)

exclude = {
    "United States",
    "Northeast",
    "Midwest",
    "South",
    "West",
    "Puerto Rico"
}

wide = wide[
    ~wide["State"].isin(exclude)
].copy()

census = wide.melt(
    id_vars="State",
    var_name="Year",
    value_name="state_population"
)

census["Year"] = pd.to_numeric(
    census["Year"],
    errors="coerce"
)

census["state_population"] = pd.to_numeric(
    census["state_population"],
    errors="coerce"
)

census = census[
    census["Year"].between(2013, 2023)
    & census["State"].notna()
    & census["state_population"].notna()
].copy()

census["Year"] = census["Year"].astype(int)

census = (
    census[
        [
            "State",
            "Year",
            "state_population"
        ]
    ]
    .drop_duplicates(["State", "Year"])
    .sort_values(["State", "Year"])
    .reset_index(drop=True)
)

print("Census shape:", census.shape)
print("States:", census["State"].nunique())
print("Years:", sorted(census["Year"].unique()))

Census shape: (561, 3)
States: 51
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [10]:
# validate census data

if census.duplicated(["State", "Year"]).any():
    raise ValueError("Census contains duplicate State-Year rows.")

if (census["state_population"] <= 0).any():
    raise ValueError("Census contains non-positive population values.")

print("Census validation passed.")

Census validation passed.


In [11]:
# merge Google, HCUP, and census data sets, using state & year
dataset = (
    google_year
    .merge(
        hcup,
        on=["State", "Year"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        census,
        on=["State", "Year"],
        how="inner",
        validate="one_to_one",
    )
)

print("Merged shape:", dataset.shape)
print("States:", dataset["State"].nunique())
print("Years:", sorted(dataset["Year"].unique()))

Merged shape: (496, 20)
States: 46
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [12]:
# calculate admission rates

dataset["admission_rate_per_100k"] = (
    dataset["mental_health_admissions"]
    / dataset["state_population"]
    * 100_000
)

dataset["admission_rate_per_100k"].describe()

count    496.000000
mean     404.481477
std      148.571382
min      110.110717
25%      286.950953
50%      396.701019
75%      500.600288
max      826.908647
Name: admission_rate_per_100k, dtype: float64

In [13]:
# keep predictive modeling period

dataset = dataset[
    dataset["Year"].between(2014, 2023)
].copy()

dataset = dataset.sort_values(
    ["State", "Year"]
).reset_index(drop=True)

print("Final modeling period:", dataset["Year"].min(), "to", dataset["Year"].max())

Final modeling period: 2014 to 2023


In [14]:
#select final variables

lag_columns = [
    f"{term}_lag1"
    for term in SEARCH_TERMS
]

final_columns = [
    "State",
    "Year",
    "state_population",
    "mental_health_admissions",
    "admission_rate_per_100k",
    *lag_columns,
]

dataset = dataset[final_columns].copy()

dataset.head()

,State,Year,state_population,mental_health_admissions,admission_rate_per_100k,suicidal thoughts_lag1,suicide hotline_lag1,self harm_lag1,mental health crisis_lag1,suicide prevention_lag1,crisis hotline_lag1,psychiatric hospital_lag1,depression help_lag1
0,Alaska,2015,738430.0,2200.0,297.929391,0.000000,8.083333,0.000000,NaN,12.416667,NaN,0.000000,0.000000
1,Alaska,2016,742575.0,2150.0,289.533044,0.000000,0.000000,16.000000,NaN,9.250000,NaN,0.000000,0.000000
2,Alaska,2017,740983.0,2250.0,303.650691,0.000000,0.000000,6.333333,NaN,12.750000,NaN,0.000000,13.250000
3,Alaska,2018,736624.0,2350.0,319.023002,7.666667,24.333333,7.500000,NaN,3.666667,NaN,0.000000,6.416667
4,Alaska,2019,733603.0,2450.0,333.968100,0.000000,22.000000,0.000000,NaN,18.333333,NaN,8.333333,5.583333


In [15]:
# validate final data set

expected_years = set(range(2014, 2024))

print("=" * 70)
print("FINAL DATASET VALIDATION")
print("=" * 70)

print("\nShape:")
print(dataset.shape)

print("\nStates:")
print(dataset["State"].nunique())

print("\nYears:")
print(sorted(dataset["Year"].unique()))

print("\nDuplicate State-Year rows:")
print(
    dataset.duplicated(
        ["State", "Year"]
    ).sum()
)

print("\nMissing values:")
print(dataset.isna().sum())

print("\nAdmission distribution:")
print(
    dataset["mental_health_admissions"].describe()
)

print("\nAdmission rate distribution:")
print(
    dataset["admission_rate_per_100k"].describe()
)

FINAL DATASET VALIDATION

Shape:
(452, 13)

States:
46

Years:
[np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Duplicate State-Year rows:
0

Missing values:
State                         0
Year                          0
state_population              0
mental_health_admissions      0
admission_rate_per_100k       0
suicidal thoughts_lag1       10
suicide hotline_lag1          0
self harm_lag1                0
mental health crisis_lag1    19
suicide prevention_lag1       0
crisis hotline_lag1          19
psychiatric hospital_lag1    10
depression help_lag1          0
dtype: int64

Admission distribution:
count       452.000000
mean      26961.061947
std       29354.382407
min        1150.000000
25%        8337.500000
50%       16050.000000
75%       33562.500000
max      145100.000000
Name: mental_health_admissions, dtype: float64

Admission rate distribution:
count    452.0

In [16]:
#check state-year structure, identify incomplete

state_year_check = (
    dataset
    .groupby("State")["Year"]
    .agg(["min", "max", "count"])
)

print(state_year_check)

if dataset.duplicated(["State", "Year"]).any():
    raise ValueError(
        "Final dataset contains duplicate State-Year observations."
    )

if dataset["state_population"].isna().any():
    raise ValueError(
        "Final dataset contains missing population values."
    )

if dataset["mental_health_admissions"].isna().any():
    raise ValueError(
        "Final dataset contains missing outcome values."
    )

if (dataset["state_population"] <= 0).any():
    raise ValueError(
        "Final dataset contains invalid population values."
    )

if (dataset["mental_health_admissions"] < 0).any():
    raise ValueError(
        "Final dataset contains negative admission values."
    )

print("\nFinal validation passed.")

                 min   max  count
State                            
Alaska          2015  2023      9
Arizona         2014  2023     10
Arkansas        2014  2023     10
California      2014  2022      9
Colorado        2014  2023     10
Delaware        2016  2023      8
Florida         2014  2023     10
Georgia         2014  2023     10
Hawaii          2014  2023     10
Illinois        2014  2023     10
Indiana         2014  2023     10
Iowa            2014  2023     10
Kansas          2014  2023     10
Kentucky        2014  2023     10
Louisiana       2014  2023     10
Maine           2014  2023     10
Maryland        2014  2023     10
Massachusetts   2014  2023     10
Michigan        2014  2023     10
Minnesota       2014  2023     10
Mississippi     2014  2023     10
Missouri        2014  2023     10
Montana         2014  2023     10
Nebraska        2014  2023     10
Nevada          2014  2020      7
New Jersey      2014  2023     10
New Mexico      2014  2023     10
New York      

In [17]:
# check final data set

display(
    dataset
    .sort_values(["State", "Year"])
    .head(25)
)

,State,Year,state_population,mental_health_admissions,admission_rate_per_100k,suicidal thoughts_lag1,suicide hotline_lag1,self harm_lag1,mental health crisis_lag1,suicide prevention_lag1,crisis hotline_lag1,psychiatric hospital_lag1,depression help_lag1
0,Alaska,2015,738430.0,2200.0,297.929391,0.000000,8.083333,0.000000,NaN,12.416667,NaN,0.000000,0.000000
1,Alaska,2016,742575.0,2150.0,289.533044,0.000000,0.000000,16.000000,NaN,9.250000,NaN,0.000000,0.000000
2,Alaska,2017,740983.0,2250.0,303.650691,0.000000,0.000000,6.333333,NaN,12.750000,NaN,0.000000,13.250000
3,Alaska,2018,736624.0,2350.0,319.023002,7.666667,24.333333,7.500000,NaN,3.666667,NaN,0.000000,6.416667
4,Alaska,2019,733603.0,2450.0,333.968100,0.000000,22.000000,0.000000,NaN,18.333333,NaN,8.333333,5.583333
5,Alaska,2020,731158.0,2550.0,348.761827,0.000000,11.416667,0.000000,NaN,27.500000,NaN,0.000000,10.583333
6,Alaska,2021,734590.0,2800.0,381.165004,0.000000,14.500000,0.000000,NaN,11.916667,NaN,0.000000,8.333333
7,Alaska,2022,733659.0,2900.0,395.279006,0.000000,7.000000,0.000000,NaN,10.000000,NaN,0.000000,0.000000
8,Alaska,2023,734654.0,2750.0,374.325873,0.000000,15.333333,15.916667,NaN,11.500000,NaN,0.000000,3.750000
9,Arizona,2014,6732873.0,15050.0,223.530133,67.750000,26.416667,69.833333,4.833333,21.083333,6.333333,50.083333,53.416667


In [18]:
# save to processed data

dataset.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {len(dataset.columns)}")

Saved: /Users/caseybrookshier/Team-Rho-Capstone-Project/data/processed/team_rho_state_year_prediction_dataset.csv
Rows: 452
Columns: 13


In [19]:
# confirm saved file available

check = pd.read_csv(OUTPUT_FILE)

print("Saved file:", OUTPUT_FILE)
print("Shape:", check.shape)
print("Duplicate State-Year rows:", check.duplicated(["State", "Year"]).sum())

display(check.head())

Saved file: /Users/caseybrookshier/Team-Rho-Capstone-Project/data/processed/team_rho_state_year_prediction_dataset.csv
Shape: (452, 13)
Duplicate State-Year rows: 0


,State,Year,state_population,mental_health_admissions,admission_rate_per_100k,suicidal thoughts_lag1,suicide hotline_lag1,self harm_lag1,mental health crisis_lag1,suicide prevention_lag1,crisis hotline_lag1,psychiatric hospital_lag1,depression help_lag1
0,Alaska,2015,738430.0,2200.0,297.929391,0.000000,8.083333,0.000000,NaN,12.416667,NaN,0.000000,0.000000
1,Alaska,2016,742575.0,2150.0,289.533044,0.000000,0.000000,16.000000,NaN,9.250000,NaN,0.000000,0.000000
2,Alaska,2017,740983.0,2250.0,303.650691,0.000000,0.000000,6.333333,NaN,12.750000,NaN,0.000000,13.250000
3,Alaska,2018,736624.0,2350.0,319.023002,7.666667,24.333333,7.500000,NaN,3.666667,NaN,0.000000,6.416667
4,Alaska,2019,733603.0,2450.0,333.968100,0.000000,22.000000,0.000000,NaN,18.333333,NaN,8.333333,5.583333


In [20]:
# final summary

print("=" * 70)
print("TEAM RHO DATASET BUILD COMPLETE")
print("=" * 70)

print(f"Output: {OUTPUT_FILE.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {len(dataset.columns)}")
print(f"States: {dataset['State'].nunique()}")
print(f"Years: {dataset['Year'].min()}-{dataset['Year'].max()}")
print(
    f"Duplicate State-Year rows: "
    f"{dataset.duplicated(['State', 'Year']).sum()}"
)
print(
    f"Missing outcome values: "
    f"{dataset['mental_health_admissions'].isna().sum()}"
)
print(
    f"Missing population values: "
    f"{dataset['state_population'].isna().sum()}"
)

TEAM RHO DATASET BUILD COMPLETE
Output: data/processed/team_rho_state_year_prediction_dataset.csv
Rows: 452
Columns: 13
States: 46
Years: 2014-2023
Duplicate State-Year rows: 0
Missing outcome values: 0
Missing population values: 0
